# Обучение кризисного классификатора

|  |  |
|---|---|
| Данные | Синтетические реплики из `crisis_train.yaml` |
| Модель | Векторы rubert-tiny2 и логистическая регрессия |
| Ограничение | До 3 % ложных тревог при подборе порога |

Чат проверяет каждую реплику. Низкий порог помогает заметить больше кризисных сообщений, но обычный ответ на вопрос о самочувствии тоже может прервать диалог. Подберём порог на обучающей выборке и измерим полноту.

## 1. Подготовка данных

Примеры сгенерированы локальной языковой моделью и отобраны по смыслу. В группу без кризиса входят в том числе короткие ответы на уточняющие вопросы. Перед делением выборки удаляем повторяющиеся тексты, чтобы одна и та же реплика не участвовала и в обучении, и в оценке одной модели.

In [1]:
import json
from pathlib import Path
import sys
from typing import Final, Literal, cast

import numpy as np
from numpy.typing import NDArray
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score
from sklearn.model_selection import StratifiedKFold
import yaml

from anamnesis.domain.types import LanguageCode
from anamnesis.infrastructure.safety.encoder_detector import ClassifierFile, EncoderCrisisDetector
from anamnesis.infrastructure.safety.sentence_encoder import SentenceEncoder, Vectors
from anamnesis.main.config.config import Config
from anamnesis.main.config.knowledge import KnowledgeConfig


if Path.cwd().name == "benchmarks":
    sys.path.insert(0, str(Path.cwd().parent))

from benchmarks.utils.support import load_data, make_table, use_project_root


use_project_root()

ru: Final[LanguageCode] = LanguageCode("ru")
config: KnowledgeConfig = Config.load().knowledge
encoder_name: Final[str] = "rubert-tiny2"
seed: Final[int] = 42
max_false_rate: Final[float] = 0.03

type TrainingGroups = dict[Literal["crisis", "safe"], dict[str, list[str]]]

groups: TrainingGroups = cast(
    TrainingGroups,
    load_data("crisis_train.yaml"),
)
data: pd.DataFrame = pd.DataFrame(
    [
        (text.strip(), label)
        for label, subgroups in groups.items()
        for texts in subgroups.values()
        for text in texts
    ],
    columns=["Текст", "Класс"],
)
assert set(data["Класс"]) == {"crisis", "safe"}
assert data.groupby("Текст")["Класс"].nunique().eq(1).all()
unique: pd.DataFrame = data.drop_duplicates("Текст")
pd.concat(
    [data["Класс"].value_counts(), unique["Класс"].value_counts()],
    axis=1,
    keys=["До удаления повторов", "Для обучения"],
)

,До удаления повторов,Для обучения
Класс,,
safe,739,738
crisis,194,193


Энкодер превращает текст в вектор единичной длины. На векторах целых реплик обучим логистическую регрессию. Для проверки длинной реплики возьмём также векторы отдельных предложений. Так одно тревожное предложение может определить оценку всего текста, как при обработке сообщения в чате.

In [2]:
type Indices = NDArray[np.intp]

encoder: SentenceEncoder = SentenceEncoder.load(config.models_dir / encoder_name)
segments: list[tuple[str, ...]] = [
    EncoderCrisisDetector._segments(text) for text in unique["Текст"]
]
texts: list[str] = list(dict.fromkeys(text for parts in segments for text in parts))
batch_size: Final[int] = 32
vectors: Vectors = np.vstack(
    [
        encoder.encode(texts[start : start + batch_size])
        for start in range(0, len(texts), batch_size)
    ]
)
positions: dict[str, int] = {text: i for i, text in enumerate(texts)}
indices: list[Indices] = [np.array([positions[text] for text in parts]) for parts in segments]
whole: Vectors = vectors[[parts[0] for parts in indices]]
labels: NDArray[np.bool_] = unique["Класс"].eq("crisis").to_numpy()

## 2. Подбор параметров

Делим реплики на пять частей с близкими долями классов. Каждую часть по очереди откладываем и обучаем модель на остальных четырёх. Каждая реплика получает оценку от модели, которая её не видела. Для составной реплики берём наибольшую вероятность по её фрагментам.

Параметр $C$ определяет, насколько свободно модель подбирает веса. Чем меньше $C$, тем сильнее штраф за большие веса. Сравним четыре значения по средней точности (average precision). Она выше, если кризисные реплики получают более высокие оценки, чем остальные.

In [3]:
def fit_model(c_value: int, inputs: Vectors, targets: NDArray[np.bool_]) -> LogisticRegression:
    model: LogisticRegression = LogisticRegression(
        C=c_value,
        class_weight="balanced",
        max_iter=5000,
        random_state=seed,
    )
    return model.fit(inputs, targets)


splitter: StratifiedKFold = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
folds: list[tuple[Indices, Indices]] = list(splitter.split(whole, labels))
candidates: dict[int, NDArray[np.float64]] = {}
c_value: int
train: Indices
test: Indices
for c_value in [1, 10, 100, 1000]:
    scores: NDArray[np.float64] = np.empty(len(labels))
    for train, test in folds:
        model: LogisticRegression = fit_model(c_value, whole[train], labels[train])
        parts: list[Indices] = [indices[i] for i in test]
        test_vectors: Vectors = vectors[np.concatenate(parts)]
        probabilities: NDArray[np.float64] = model.predict_proba(test_vectors)[:, 1]
        starts: Indices = np.cumsum([0] + [len(part) for part in parts[:-1]])
        scores[test] = np.maximum.reduceat(probabilities, starts)
    candidates[c_value] = scores

quality: pd.Series = pd.Series(
    {c: average_precision_score(labels, scores) for c, scores in candidates.items()},
    name="Average precision",
).rename_axis("C")
best_c: int = int(quality.idxmax())
quality.round(4).to_frame()

,Average precision
C,
1,0.8833
10,0.9016
100,0.8878
1000,0.8635


Сортируем оценки реплик без кризиса по убыванию. Разрешаем столько ложных тревог, сколько укладывается в 3 % выборки этого класса. Порог ставим чуть выше оценки следующей реплики. Если несколько реплик получили граничную оценку, все они останутся ниже порога.

In [4]:
best_scores: NDArray[np.float64] = candidates[best_c]
safe: NDArray[np.float64] = np.sort(best_scores[~labels])[::-1]
allowed: int = int(max_false_rate * len(safe))
threshold: float = float(np.nextafter(np.float32(safe[allowed]), np.float32(1)))
predicted: NDArray[np.bool_] = best_scores >= threshold
assert predicted[~labels].sum() <= allowed

make_table(
    {
        "C": best_c,
        "Порог": threshold,
        "Кризисных найдено": predicted[labels].sum(),
        "Кризисных пропущено": (~predicted[labels]).sum(),
        "Полнота": predicted[labels].mean(),
        "Ложные тревоги": predicted[~labels].sum(),
        "Доля ложных тревог": predicted[~labels].mean(),
    },
    decimals=4,
)

,C,Порог,Кризисных найдено,Кризисных пропущено,Полнота,Ложные тревоги,Доля ложных тревог
0,10,0.7486,144,49,0.7461,22,0.0298


## 3. Экспорт параметров

Обучим модель с выбранным $C$ на всех уникальных репликах. Сохраним имя энкодера, веса, свободный член и порог. По умолчанию файл записывается в `resources/ru/crisis_classifier.yaml`. Проверим, что детектор сервиса загружает сохранённые параметры.

In [5]:
final: LogisticRegression = fit_model(best_c, whole, labels)
parameters: ClassifierFile = ClassifierFile(
    encoder=encoder_name,
    threshold=threshold,
    intercept=float(final.intercept_[0]),
    weights=tuple(final.coef_[0].tolist()),
)

path: Path = config.resources_dir / ru / "crisis_classifier.yaml"
path.parent.mkdir(parents=True, exist_ok=True)
source_path: Path = config.models_dir / encoder_name / "source.json"
source: dict[str, str] = json.loads(source_path.read_text(encoding="utf-8"))
header: str = (
    f"# {source['repository']}@{source['revision']}\n"
    f"# benchmarks/03_train_crisis_classifier.ipynb, "
    f"{len(labels)} реплик, C = {best_c}, seed = {seed}\n"
)
path.write_text(
    header + yaml.safe_dump(parameters.model_dump(mode="json"), sort_keys=False),
    encoding="utf-8",
)
EncoderCrisisDetector.load(config.resources_dir, (ru,), config.models_dir)

make_table(
    {
        "Файл": str(path.relative_to(config.resources_dir)),
        "Энкодер": parameters.encoder,
        "Веса загружены": len(parameters.weights),
    }
)

,Файл,Энкодер,Веса загружены
0,ru/crisis_classifier.yaml,rubert-tiny2,312


## Выводы

1. После удаления двух повторов осталось 193 кризисных реплики и 738 реплик без кризиса. На пяти частях выборки лучший результат получился при $C=10$ (average precision 0,902).
2. При пороге 0,7486 обнаружены 144 кризисные реплики из 193. Из 738 реплик без кризиса 22 дали ложную тревогу (2,98 %). Порог укладывается в выбранное ограничение, но 49 кризисных реплик остаются ниже него.
3. Детектор загрузил итоговую модель с 312 весами. Порог подбирался на тех же пяти частях, поэтому для оценки качества на новых формулировках используется отдельный набор из второго ноутбука.